# 05 — V3 camera robustness and calibration-gain sensitivity
Run after Notebook 04 v3 has completed all six corrected model/seed evaluations. Use the same
`wildlife_confidence_project_v3` root and `five_categories_detected_only_v3` experiment.
The input evaluation is `evaluation_v3`; audit outputs go only to `camera_audit_v3`.

This notebook verifies the corrected Notebook 03 selected-checkpoint metadata and Notebook 04's
completed evaluation, then audits the existing saved test predictions. It first reproduces the
primary event-weighted NLL comparison, then adds equal-camera weighting, camera-level behavior,
abstention summaries, and leave-one-camera-out architecture influence.

No retraining, temperature/threshold refitting, data changes, image review, downloads or model
inference occur here. Keep V1/V2 files and the completed V3 Notebooks 01–04 unchanged. This is sensitivity analysis
of the same test population, not independent confirmation. Do not tune methods or remove cameras
using its outcomes. Prior V1/V2 analysis informed the V3 design; do not imply independent preregistration.


**Detected-only population and prior test exposure.** All five V3 partitions contain only the retained V2 animal crops; no full-frame fallback is used for fitting or evaluation. Every reported coverage fraction is conditional on this retained population, rather than an unfiltered camera stream. Original development camera caps applied before exclusion; post-exclusion shares can change. Camera assignments are inherited unchanged, and V3 was designed after V1/V2 test inspection. The test cameras remain separate from V3 classifier fitting, but are not a fresh independent confirmation. Detection success does not establish correct species labels or crop quality.


**V3 scope:** full-frame fallbacks are excluded from training, validation, calibration, policy and test.
Detected crops may still contain incorrect regions or species; file integrity does not prove semantic quality.
We retain V2 camera assignments, so prior V2 test exposure remains a limitation. Original development
caps apply before exclusion; post-exclusion shares are reported. Coverage is conditional on detection.


## 1. Setup and fixed analysis choices
NLL remains the primary calibration-gain metric from Notebook 04. Brier and ECE are supporting
outcomes. The original valid weighting calculations are retained. ECE bins, camera-bootstrap seed
and replicate count are inherited from the verified parent evaluation, so the event-weighted primary
comparison must reproduce its point estimates and interval endpoints before sensitivity results proceed.

Event weighting gives each selected event equal weight. Equal-camera weighting gives each observed
camera equal total weight, using event weights proportional to 1/n_camera. These are distinct
populations, not alternative ways to obtain a preferred score. Small-camera flags remain descriptive.
Intervals are conditional, descriptive percentile intervals; supporting comparisons are exploratory.

NLL/Brier/ECE use all test events, whether or not a selective policy accepts them. A policy
that abstains everywhere therefore does not prevent the main calibration-gain comparison.
Acceptance affects coverage and accepted-error summaries, not which events contribute to those
probability-quality metrics. Agreement with Notebook 04 checks the same saved predictions and
analysis; it is not an additional independent validation sample.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import io, json, hashlib, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = None
EXPERIMENT_ID = 'five_categories_detected_only_v3'
TRAIN_ID = 'training_v3'
if PROJECT_ROOT is None:
    found = list(dict.fromkeys(p.resolve() for p in [Path.cwd()/'wildlife_confidence_project_v3', Path.cwd()]
        if (p/'reports'/EXPERIMENT_ID/'evaluation_v3/completion.json').is_file()))
    if len(found) != 1:
        raise FileNotFoundError('Set PROJECT_ROOT to the prepared v3 project; complete Notebook 04 v3 first.')
    ROOT = found[0]
else:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
PREPARED = ROOT/'data/prepared'
EXP = PREPARED/EXPERIMENT_ID
TRAIN_REPORT = ROOT/'reports'/EXPERIMENT_ID/TRAIN_ID
PREVIOUS = ROOT/'reports'/EXPERIMENT_ID/'evaluation_v3'
REPORT = ROOT/'reports'/EXPERIMENT_ID/'camera_audit_v3'
REPORT.mkdir(parents=True, exist_ok=True)
CFG = {'audit_version':3, 'small_camera_flag_below':30, 'primary_gain_metric':'nll',
       'analysis_status':'v2_camera_sensitivity_existing_predictions',
       'gain_direction':'raw minus scaled; positive means calibration improves loss',
       'interaction_direction':'EfficientNet gain minus MobileNet gain; positive favours EfficientNet'}
# ECE bins and bootstrap seed/repeats are inherited from the verified parent evaluation below.

def now():
    return datetime.now(timezone.utc).isoformat()

def digest(payload):
    return hashlib.sha256(payload).hexdigest()

def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

def require(condition, message):
    if not condition:
        raise ValueError(message)

def contained(base, relative):
    require(isinstance(relative, str), 'Expected a relative path string')
    relative = Path(relative)
    require(not relative.is_absolute() and '..' not in relative.parts, 'Unsafe relative input path')
    base = Path(base).resolve()
    path = (base/relative).resolve()
    require(path.is_relative_to(base), 'Input path leaves its expected directory')
    return path

def safe(relative):
    return contained(ROOT, relative)

def verified_bytes(path, expected=None):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Missing audit input: {path}')
    payload = path.read_bytes()
    require(expected is None or digest(payload) == expected, f'Input hash mismatch: {path}')
    return payload

def json_input(path, expected=None):
    value = json.loads(verified_bytes(path, expected).decode('utf-8'))
    require(isinstance(value, dict), f'Expected a JSON object: {path}')
    return value

def jtext(value):
    return json.dumps(value, indent=2, sort_keys=True, ensure_ascii=False, allow_nan=False)+'\n'

def write(path, text):
    path = Path(path)
    temporary = Path(str(path)+'.tmp')
    temporary.write_text(text, encoding='utf-8')
    temporary.replace(path)

def freeze(path, text):
    if path.exists() and path.read_text(encoding='utf-8') != text:
        raise ValueError(f'Audit inputs/settings differ: {path}. Preserve the previous audit.')
    if not path.exists():
        write(path, text)

def table(name, rows):
    frame = rows if isinstance(rows, pd.DataFrame) else pd.DataFrame(rows)
    write(REPORT/name, frame.to_csv(index=False))
    return frame

print('V3 project:', ROOT, '| audit only: existing predictions, no training or inference')


## 2. Verify completed v3 inputs before computing the audit
Explicit exceptions protect the experiment identity, completed evaluation output hashes, frozen
selection/manifests, corrected training index and selected-checkpoint metadata, and all six test-logit
snapshots. The audit reads the same bytes it hashes and preserves original prediction-hash records.
It requires all three planned seeds, both architectures, and raw/scaled policy records, including
`insufficient_evidence` and no-threshold outcomes. It does not need PyTorch or load model weights.

Notebook 04 selects using its frozen binomial upper bound and unchanged support requirements.
A selected policy may have zero observed policy errors and unavailable camera uncertainty; camera
bootstrap estimates are diagnostics only and are not re-applied as selection gates here.

**Inherited binomial rule.** Notebook 04 uses a one-sided Clopper–Pearson upper bound with
$\alpha_*=0.05/(12\times85)$, requiring the bound to be at most 5%, at least 100 accepted
policy events and at least five accepting policy cameras. It maximizes accepted count, with
the higher threshold breaking ties. For zero policy errors the bound is $1-\alpha_*^{1/n}$,
so sufficiently supported zero-error candidates can be selected. Notebook 04 gives the full
derivation and assumptions; this is not a guaranteed camera-aware error bound.

A selected threshold with `camera_uncertainty_available=False` and a missing camera upper
estimate is valid under that rule. An estimate above 5% also does not reverse selection.
Consult Notebook 04's policy summary, candidate CSVs and frozen decisions for these
**policy-camera diagnostics**. The camera performance and bootstrap analyses below concern
**test cameras**; they do not re-estimate or approve the policy threshold.

The amended selector reports `no_qualifying_threshold` if no candidate meets its binomial
and support conditions. Validation also understands legacy `insufficient_evidence` records;
unavailable camera diagnostics alone do not cause rejection in the amended rule. Only a
complete, internally matching parent evaluation and its original hashes are accepted;
records from different frozen evaluation methods must not be combined.


In [ ]:
required_parent_outputs = {'evaluation_protocol.json', 'frozen_calibration_and_policies.json',
    'test_exports.json', 'test_results.csv', 'calibration_gain_comparison.csv', 'test_seed_summary.csv'}
completion_payload = verified_bytes(PREVIOUS/'completion.json')
PARENT_COMPLETION_SHA = digest(completion_payload)
completion = json.loads(completion_payload.decode('utf-8'))
require(completion.get('experiment_id') == EXPERIMENT_ID and completion.get('completed_runs') == 6
        and completion.get('test_used_for_fitting') is False, 'Not a completed six-run v3 evaluation')
require(isinstance(completion.get('outputs'), dict) and required_parent_outputs.issubset(completion['outputs']),
        'Notebook 04 completion record is missing required audit inputs')

def verify_parent_unchanged():
    verified_bytes(PREVIOUS/'completion.json', PARENT_COMPLETION_SHA)
    for name, expected_hash in completion['outputs'].items():
        require(Path(name).name == name, 'Parent output name must be a filename')
        verified_bytes(contained(PREVIOUS, name), expected_hash)

verify_parent_unchanged()

def parent_json(name):
    return json_input(PREVIOUS/name, completion['outputs'][name])

def parent_csv(name):
    return pd.read_csv(io.BytesIO(verified_bytes(PREVIOUS/name, completion['outputs'][name])))

eval_protocol = parent_json('evaluation_protocol.json')
decisions = parent_json('frozen_calibration_and_policies.json')
exports = parent_json('test_exports.json')
require(completion['evaluation_sha256'] == completion['outputs']['evaluation_protocol.json']
        and completion['decisions_sha256'] == completion['outputs']['frozen_calibration_and_policies.json'],
        'Parent evaluation/decision digests are inconsistent')
require(eval_protocol.get('experiment_id') == EXPERIMENT_ID and eval_protocol.get('training_id') == TRAIN_ID,
        'Parent evaluation is not linked to corrected Notebook 03 v3')
require(decisions.get('evaluation_sha256') == completion['evaluation_sha256']
        and exports.get('evaluation_sha256') == completion['evaluation_sha256']
        and exports.get('decisions_sha256') == completion['decisions_sha256'], 'Prediction/decision evaluation mismatch')
settings = eval_protocol['settings']
require(settings.get('primary_calibration_gain_metric') == 'nll'
        and completion.get('primary_calibration_gain_metric') == 'nll', 'Parent primary metric is not NLL')
for name in ['ece_bins', 'bootstrap_repeats', 'bootstrap_seed']:
    require(isinstance(settings.get(name), int), f'Invalid parent bootstrap/metric setting: {name}')
    CFG[name] = settings[name]
require(CFG['ece_bins'] > 0 and CFG['bootstrap_repeats'] >= 2, 'Invalid parent metric/bootstrap sizes')

protocol = json_input(EXP/'protocol.json', eval_protocol['protocol_sha256'])
selection = json_input(EXP/'selection_protocol.json', eval_protocol['selection_protocol_sha256'])
camera_record = json_input(PREPARED/'camera_split.json', eval_protocol['camera_split_sha256'])
require(protocol.get('pipeline_kind') == 'megadetector_detected_only_single_v3'
        and protocol.get('protocol_version') == 3 and protocol.get('experiment_id') == EXPERIMENT_ID,
        'Input preprocessing protocol is not v3')
require(selection.get('selection_version') == 3 and selection.get('experiment_id') == EXPERIMENT_ID
        and camera_record.get('protocol') == 'camera_stratified_event_v2'
        and camera_record.get('experiment_id') == "five_categories_camera_balanced_v2", 'Wrong v3 selection/camera experiment')
for name, expected_hash in protocol['input_sha256'].items():
    verified_bytes(contained(PREPARED, name), expected_hash)
classes = json_input(EXP/'classes.json')
CLASSES = classes['classes']
K = len(CLASSES)
require(CLASSES == ['coyote','deer','fox','rabbit','skunk']
        and classes['class_to_idx'] == {name:i for i,name in enumerate(CLASSES)}
        and protocol['classes'] == CLASSES and selection['classes'] == CLASSES,
        'Class order/index mismatch')
manifest = pd.read_csv(io.BytesIO(verified_bytes(EXP/'training_ready_manifest.csv',
    eval_protocol['ready_manifest_sha256'])), dtype=str, keep_default_na=False)
base = pd.read_csv(io.BytesIO(verified_bytes(EXP/'subset_manifest.csv',
    eval_protocol['subset_sha256'])), dtype=str, keep_default_na=False)
require(set(base.columns).issubset(manifest.columns) and manifest[base.columns].equals(base),
        'Ready manifest differs from the frozen v3 sample')
manifest['label'] = manifest.label.astype(int)
require(manifest.image_id.is_unique and manifest.seq_id.is_unique
        and manifest.groupby('location').split.nunique().eq(1).all()
        and manifest.label.eq(manifest.class_name.map({name:i for i,name in enumerate(CLASSES)})).all(),
        'Duplicate events, camera leakage or invalid labels')
require(set(manifest.split) == {'train','validation','calibration','policy','test'}
        and manifest.location.map(camera_record['assignments']).eq(manifest.split).all(),
        'Camera assignment or split mismatch')


# V3-specific safeguard: the selected set must be EXACTLY V2's animal-crop subset.
# No retained fallback, top-up, changed camera assignment, label or replacement frame is allowed.
crop_plan_v3 = pd.read_csv(EXP / 'crop_plan.csv', dtype=str, keep_default_na=False)
parent_plan_v2 = pd.read_csv(PREPARED / 'v2_reference/crop_plan.csv', dtype=str, keep_default_na=False)
parent_base_v2 = pd.read_csv(PREPARED / 'v2_reference/subset_manifest.csv', dtype=str, keep_default_na=False)
parent_ordered_plan = parent_plan_v2.set_index('image_id').loc[parent_base_v2.image_id]
expected_crop_ids = parent_base_v2.loc[parent_ordered_plan['mode'].eq('animal_crop').to_numpy(), 'image_id'].tolist()
if (not crop_plan_v3.image_id.is_unique or not crop_plan_v3['mode'].eq('animal_crop').all()
        or crop_plan_v3.image_id.tolist() != expected_crop_ids
        or manifest.image_id.tolist() != expected_crop_ids):
    raise ValueError('V3 contains a fallback, replacement event, missing crop, or changed row order.')
original_rows = parent_base_v2.set_index('image_id').loc[expected_crop_ids]
for column in ['file_name','location','seq_id','category_id','class_name','label','split']:
    if manifest[column].astype(str).tolist() != original_rows[column].astype(str).tolist():
        raise ValueError(f'V3 changed inherited event identity: {column}')
if protocol['crop_rule'].get('no_detection') != 'exclude_event_no_replacement':
    raise ValueError('V3 must exclude no-detection events without replacement.')

train_cfg = json_input(TRAIN_REPORT/'training_protocol.json', eval_protocol['training_sha256'])
index = json_input(TRAIN_REPORT/'training_index.json')
require(train_cfg.get('training_id') == TRAIN_ID and index.get('training_id') == TRAIN_ID
        and train_cfg.get('experiment_id') == EXPERIMENT_ID and index.get('experiment_id') == EXPERIMENT_ID
        and index.get('all_planned_runs_complete') is True and index.get('test_predictions_computed') is False,
        'Training records are not the completed corrected v3 experiment')
require(train_cfg.get('selection') == 'lowest_validation_NLL_across_head_and_finetune'
        and train_cfg.get('selection_weighting') == 'event'
        and train_cfg.get('final_checkpoint_file') == 'best_selected.pt'
        and train_cfg.get('classes') == CLASSES and index.get('classes') == CLASSES,
        'Unexpected selected-checkpoint method or classes')
for field in ['training_sha256','protocol_sha256','subset_sha256','camera_split_sha256','selection_protocol_sha256']:
    require(index.get(field) == eval_protocol[field], f'Training/evaluation provenance differs: {field}')
require(train_cfg.get('ready_manifest_sha256') == eval_protocol['ready_manifest_sha256'],
        'Training ready-manifest provenance differs')
MODELS = train_cfg['models']
SEEDS = train_cfg['planned_seeds']
require(MODELS == ['mobilenet_v3_large','efficientnet_b0'] and SEEDS == [42,123,2026],
        'Unexpected model/seed plan')
EXPECTED = {(model,seed) for model in MODELS for seed in SEEDS}

def keyed_runs(rows, label):
    require(isinstance(rows, list), f'{label} must be a list')
    keyed = {}
    for row in rows:
        key = (row['model'], int(row['seed']))
        require(key in EXPECTED and key not in keyed, f'Duplicate/unexpected run in {label}')
        keyed[key] = row
    require(set(keyed) == EXPECTED, f'Missing runs in {label}')
    return keyed

TRAIN_RUNS = keyed_runs(index['completed_runs'], 'training index')
EVAL_RUNS = keyed_runs(eval_protocol['runs'], 'parent evaluation')
DEC = keyed_runs(decisions['runs'], 'frozen decisions')
EXPORT_RUNS = keyed_runs(exports['exports'], 'test-export registry')
for key, run in TRAIN_RUNS.items():
    parent = EVAL_RUNS[key]
    require(run.get('checkpoint_file') == 'best_selected.pt'
            and run.get('selected_stage') in ['head','finetune']
            and isinstance(run.get('selected_epoch'), int) and run['selected_epoch'] >= 1,
            'Invalid corrected selected-checkpoint identity')
    for field in ['checkpoint_file','checkpoint_sha256','selected_stage','selected_epoch']:
        require(parent.get(field) == run[field], f'Selected-checkpoint metadata differs: {field}')
    require(parent.get('development_exports') == run.get('exports')
            and set(run.get('exports', {})) == {'validation','calibration','policy'}
            and DEC[key]['checkpoint_sha256'] == run['checkpoint_sha256'], 'Checkpoint/export provenance mismatch')
    require(np.isfinite(DEC[key]['temperature']) and DEC[key]['temperature'] > 0, 'Invalid frozen temperature')
    require(set(DEC[key].get('policies', {})) == {'raw','scaled'}, 'Missing raw/scaled policy')
    for policy in DEC[key]['policies'].values():
        require(policy.get('status') in ['selected','no_qualifying_threshold','insufficient_evidence'], 'Unknown policy status')
        threshold = policy.get('threshold')
        if policy['status'] == 'selected':
            require(threshold is not None and np.isfinite(threshold)
                    and threshold in settings['policy_threshold_grid'], 'Invalid selected threshold')
        else:
            require(threshold is None and policy.get('accepted') == 0
                    and policy.get('accepted_error') is None, 'Abstention must not have a threshold or defined error')

test_frame = manifest.loc[manifest.split.eq('test')].reset_index(drop=True)
CAMS = np.unique(test_frame.location.to_numpy(dtype=str))
C = len(CAMS)
require(C == completion['test_cameras'] and len(test_frame) == completion['test_images'] and C >= 2,
        'Test-camera/event count mismatch')
CAM_INDEX = np.searchsorted(CAMS, test_frame.location.to_numpy(dtype=str))
N_CAM = np.bincount(CAM_INDEX, minlength=C).astype(float)
TEST = {}
input_refs = []
for key, entry in EXPORT_RUNS.items():
    path = safe(entry['file'])
    expected_path = ROOT/'models'/EXPERIMENT_ID/'evaluation_v3'/f'{key[0]}_seed{key[1]}_test_logits.npz'
    require(path == expected_path.resolve(), 'Unexpected v3 test-logit path')
    payload = verified_bytes(path, entry['sha256'])
    with np.load(io.BytesIO(payload), allow_pickle=False) as saved:
        arrays = {name:saved[name].copy() for name in saved.files}
    required = {'logits','image_ids','labels','locations','seq_ids','class_names','checkpoint_sha256',
                'evaluation_sha256','decisions_sha256','experiment_id','model_name','seed'}
    require(required.issubset(arrays), 'Test export is missing required arrays/provenance')
    require(arrays['logits'].shape == (len(test_frame), K) and np.isfinite(arrays['logits']).all(), 'Invalid test logits')
    for field, column, dtype in [('image_ids','image_id',str),('labels','label',np.int64),
                                 ('locations','location',str),('seq_ids','seq_id',str)]:
        require(np.array_equal(arrays[field], test_frame[column].to_numpy(dtype=dtype)), f'Test identity mismatch: {field}')
    require(np.array_equal(arrays['class_names'], np.asarray(CLASSES)), 'Test class order mismatch')
    for field, expected in [('evaluation_sha256',completion['evaluation_sha256']),
        ('decisions_sha256',completion['decisions_sha256']),('experiment_id',EXPERIMENT_ID),
        ('checkpoint_sha256',TRAIN_RUNS[key]['checkpoint_sha256']),('model_name',key[0]),('seed',key[1])]:
        require(arrays[field].shape == () and arrays[field].item() == expected, f'Test provenance mismatch: {field}')
    TEST[key] = arrays
    input_refs.append(entry)
old_results = parent_csv('test_results.csv')
parent_gains = parent_csv('calibration_gain_comparison.csv')
parent_seed_summary = parent_csv('test_seed_summary.csv')
require(len(old_results) == 12 and not old_results.duplicated(['model','seed','variant']).any()
        and set(zip(old_results.model,old_results.seed,old_results.variant)) ==
            {(model,seed,variant) for model,seed in EXPECTED for variant in ['raw','scaled']},
        'Parent result rows do not contain all twelve unique variants')
freeze(REPORT/'audit_protocol.json', jtext({'settings':CFG, 'experiment_id':EXPERIMENT_ID, 'training_id':TRAIN_ID,
    'parent_completion_sha256':PARENT_COMPLETION_SHA, 'parent_evaluation_sha256':completion['evaluation_sha256'],
    'decisions_sha256':completion['decisions_sha256'], 'test_exports':input_refs,
    'selected_runs':eval_protocol['runs'], 'manifest_sha256':eval_protocol['ready_manifest_sha256'],
    'primary_gain_table_sha256':completion['outputs']['calibration_gain_comparison.csv'],
    'versions':{'python':sys.version.split()[0],'numpy':np.__version__,'pandas':pd.__version__}}))
print(f'Verified {len(TEST)} v3 runs, {len(test_frame)} events and {C} test cameras; inputs remain unchanged.')


## 3. Audit representation in every split
Each row of the manifest represents one selected event, not an independent animal. We count images/events and categories at each location, show category frequencies by split, and quantify concentration.

The largest-camera share and top-three share describe how much of the image-weighted objective is contributed by a few sites. The concentration index $\sum_c p_c^2$ uses each camera's event share $p_c$. Its reciprocal is an **effective camera count based only on event-count concentration**: it equals the actual count for equally sized cameras. This is not a statistical effective sample size and does not measure habitat diversity, independence, or prediction difficulty.

For each category/split we also report the number of contributing cameras and the fraction contributed by its largest camera. This detects class–location concentration that class totals alone conceal. These diagnostics cannot identify whether a model learned background shortcuts; that requires additional evidence.


In [ ]:
SPLITS=["train","validation","calibration","policy","test"]
camera_counts=manifest.groupby(["split","location"]).size().rename("events").reset_index()
camera_counts["split_share"]=camera_counts.events/camera_counts.groupby("split").events.transform("sum")
camera_counts["small_camera"]=camera_counts.events<CFG["small_camera_flag_below"]
class_camera=manifest.groupby(["split","location","class_name"]).size().rename("events").reset_index()
class_camera["share_within_camera"]=class_camera.events/class_camera.groupby(["split","location"]).events.transform("sum")
class_camera["share_of_category_split"]=class_camera.events/class_camera.groupby(["split","class_name"]).events.transform("sum")
concentration=[];category_concentration=[]
for split in SPLITS:
    g=camera_counts.loc[camera_counts.split.eq(split)].sort_values("events",ascending=False)
    p=g.split_share.to_numpy();concentration.append({"split":split,"events":int(g.events.sum()),
        "represented_cameras":len(g),"largest_camera":g.iloc[0].location,"largest_camera_events":int(g.iloc[0].events),
        "largest_share":float(p[0]),"top_three_share":float(p[:3].sum()),
        "concentration_equivalent_cameras":float(1/(p*p).sum()),
        "small_cameras":int(g.small_camera.sum()),"minimum_events":int(g.events.min()),"maximum_events":int(g.events.max())})
    for name in CLASSES:
        h=class_camera.loc[class_camera.split.eq(split)&class_camera.class_name.eq(name)].sort_values("events",ascending=False)
        category_concentration.append({"split":split,"class":name,"events":int(h.events.sum()),"cameras":len(h),
            "largest_camera":str(h.iloc[0].location) if len(h) else None,
            "largest_camera_share":float(h.iloc[0].share_of_category_split) if len(h) else np.nan})
concentration=table("camera_concentration_by_split.csv",concentration)
table("camera_event_counts.csv",camera_counts)
table("class_camera_counts.csv",class_camera)
table("category_camera_concentration.csv",category_concentration)
class_counts=pd.crosstab(manifest.split,manifest.class_name).reindex(index=SPLITS,columns=CLASSES,fill_value=0)
class_shares=class_counts.div(class_counts.sum(axis=1),axis=0)
table("class_distribution_by_split.csv",class_shares.reset_index())
display(concentration.round(4));display(class_shares.round(4))
fig,axes=plt.subplots(1,5,figsize=(16,4))
for ax,split in zip(axes,SPLITS):
    g=camera_counts.loc[camera_counts.split.eq(split)].sort_values("events",ascending=False)
    ax.bar(np.arange(len(g)),g.split_share)
    ax.set(title=split,xlabel="Cameras ordered by count",ylabel="Share of split events",ylim=(0,1))
fig.tight_layout();fig.savefig(REPORT/"camera_concentration.png",dpi=160);plt.show();plt.close(fig)


## 4. Define weighted metrics and reproduce notebook04
For NLL, Brier and accuracy, equal-camera weighting equals the average of camera-specific metrics. For nonlinear quantities, definitions matter:

- **Equal-camera pooled ECE** applies event weights $1/n_c$ inside the same global confidence bins. It is not the mean of camera ECEs. Opposite within-bin errors at different cameras can cancel.
- **Equal-camera pooled macro-F1** is computed from a camera-weighted confusion matrix, retaining all five categories. It is not the mean of small-camera F1 scores.
- **Equal-camera accepted error** is weighted accepted mistakes divided by weighted accepted events. It describes the accepted population under the equal-camera sampling distribution. It is not the mean of conditional camera errors.

We additionally report the mean per-camera ECE and mean accepted error among cameras with any accepted events as separately labelled diagnostics. Small-camera ECE is noisy and can be biased upward. A camera with zero accepted events has undefined conditional error, not zero.

Strict all-seed accepted-error summaries are accompanied by labelled conditional means and
explicit accepting/abstaining seed IDs and counts. Camera-level tables identify zero-acceptance
locations even when the same seed accepts predictions elsewhere. No manual crop review is required.

For camera $c$, let $N_c$ be its test event count, $A_c$ its accepted count and $E_c$ its
incorrect accepted count. For $C$ observed cameras,

$$\operatorname{coverage}_{\mathrm{event}}=\frac{\sum_c A_c}{\sum_c N_c},\qquad
\operatorname{error}_{\mathrm{event}}=\frac{\sum_c E_c}{\sum_c A_c},$$

$$\operatorname{coverage}_{\mathrm{camera}}=\frac{1}{C}\sum_c\frac{A_c}{N_c},\qquad
\operatorname{error}_{\mathrm{camera}}=
\frac{\sum_c E_c/N_c}{\sum_c A_c/N_c}.$$

Each error requires a positive acceptance denominator. Equal-camera accepted error conditions
on acceptance **after** choosing a camera uniformly and an event uniformly within that camera;
it is not the unweighted average of errors among accepting cameras. A selected policy may
accept no test events on an individual camera or for an entire run; those errors remain
undefined. Accepting some events with no mistakes gives observed error zero, a different case.

The audit's strict seed-averaged `accepted_error` is undefined if any planned seed abstains
completely. `accepted_error_mean_among_accepting_seeds` is a separately labelled conditional mean
matching Notebook 04's convention, alongside contributor counts and seed IDs. A missing strict
mean and finite conditional mean can both be correct. No abstaining seed is silently assigned
zero error, and no favourable seed is chosen for the main comparison.


In [ ]:
B=CFG["ece_bins"]
METRICS=["accuracy","nll","brier","ece","macro_f1","coverage","accepted_error"]
def probs(z,T):
    v=np.asarray(z,dtype=float)/float(T);v-=v.max(1,keepdims=True)
    lp=v-np.log(np.exp(v).sum(1,keepdims=True));return np.exp(lp),lp

def camera_statistics(a,T,policy):
    z,y=a["logits"],a["labels"];p,lp=probs(z,T);pred=z.argmax(1)
    correct=pred==y;conf=p.max(1);bins=np.minimum((conf*B).astype(int),B-1)
    accept=np.zeros(len(y),dtype=bool) if policy["threshold"] is None else conf>=policy["threshold"]
    # Additive statistics: n, correct, NLL sum, Brier sum, accepted, accepted wrong,
    # followed by flattened confusion counts and confidence-bin count/confidence/correct sums.
    start=6+K*K;v=np.zeros((len(y),start+3*B))
    v[:,:6]=np.column_stack([np.ones(len(y)),correct,-lp[np.arange(len(y)),y],
                            ((p-np.eye(K)[y])**2).sum(1),accept,accept&~correct])
    v[np.arange(len(y)),6+y*K+pred]=1
    v[np.arange(len(y)),start+bins]=1
    v[np.arange(len(y)),start+B+bins]=conf
    v[np.arange(len(y)),start+2*B+bins]=correct
    agg=np.zeros((C,v.shape[1]));np.add.at(agg,CAM_INDEX,v)
    return agg

def calculate(s):
    s=np.atleast_2d(s);n=s[:,0];cm=s[:,6:6+K*K].reshape(-1,K,K)
    den=cm.sum(1)+cm.sum(2)
    f1=np.divide(2*np.diagonal(cm,axis1=1,axis2=2),den,out=np.zeros_like(den),where=den>0).mean(1)
    start=6+K*K
    ece=np.abs(s[:,start+B:start+2*B]-s[:,start+2*B:start+3*B]).sum(1)/n
    risk=np.divide(s[:,5],s[:,4],out=np.full(len(n),np.nan),where=s[:,4]>0)
    return dict(zip(METRICS,[s[:,1]/n,s[:,2]/n,s[:,3]/n,ece,f1,s[:,4]/n,risk]))

def summarize_audit_seeds(weighted_table):
    rows = []
    for (model,variant,weighting), group in weighted_table.groupby(['model','variant','weighting']):
        require(len(group) == len(SEEDS) and set(group.seed) == set(SEEDS) and not group.seed.duplicated().any(),
                'Audit seed summary is missing/duplicating a planned seed')
        contributors = group.accepted.gt(0)
        require((group.accepted_error.notna() == contributors).all(), 'Undefined accepted errors disagree with counts')
        row = {'model':model,'variant':variant,'weighting':weighting,
               **{metric:float(group[metric].mean()) if group[metric].notna().all() else np.nan for metric in METRICS},
               'seed_count':len(group),'accepted_error_contributing_seeds':int(contributors.sum()),
               'abstaining_seeds':int((~contributors).sum()),
               'accepting_seed_ids':json.dumps(sorted(group.loc[contributors,'seed'].astype(int).tolist())),
               'abstaining_seed_ids':json.dumps(sorted(group.loc[~contributors,'seed'].astype(int).tolist())),
               'seeds_with_selected_policy':int(group.policy_status.eq('selected').sum()),
               'accepted_error_mean_among_accepting_seeds':float(group.loc[contributors,'accepted_error'].mean()),
               'accepted_error_std_among_accepting_seeds':float(group.loc[contributors,'accepted_error'].std())}
        rows.append(row)
    return pd.DataFrame(rows)


def audit_abstention_tables(per_camera_table):
    seed_rows = []
    camera_rows = []
    for (model,seed,variant), group in per_camera_table.groupby(['model','seed','variant']):
        accepting = group.accepted.gt(0)
        n = int(group.accepted.sum())
        errors = int(group.accepted_errors.sum())
        seed_rows.append({'model':model,'seed':int(seed),'variant':variant,
            'policy_status':group.policy_status.iloc[0],'policy_reason':group.policy_reason.iloc[0],
            'total_cameras':len(group),'cameras_with_acceptance':int(accepting.sum()),
            'cameras_without_acceptance':int((~accepting).sum()),
            'accepting_camera_ids':json.dumps(sorted(group.loc[accepting,'location'].astype(str).tolist())),
            'abstaining_camera_ids':json.dumps(sorted(group.loc[~accepting,'location'].astype(str).tolist())),
            'accepted':n,'accepted_errors':errors,'accepted_error':errors/n if n else np.nan})
    for (model,variant,camera), group in per_camera_table.groupby(['model','variant','location']):
        require(len(group) == len(SEEDS) and set(group.seed) == set(SEEDS), 'Incomplete camera seed coverage')
        accepting = group.accepted.gt(0)
        camera_rows.append({'model':model,'variant':variant,'location':str(camera),
            'events':int(group.events.iloc[0]),'seed_count':len(group),
            'seeds_with_acceptance':int(accepting.sum()),'abstaining_seeds':int((~accepting).sum()),
            'accepting_seed_ids':json.dumps(sorted(group.loc[accepting,'seed'].astype(int).tolist())),
            'abstaining_seed_ids':json.dumps(sorted(group.loc[~accepting,'seed'].astype(int).tolist())),
            'mean_error_among_accepting_seeds':float(group.loc[accepting,'accepted_error'].mean())})
    return pd.DataFrame(seed_rows), pd.DataFrame(camera_rows)


AGG={};per_camera_rows=[];summary_rows=[];diagnostic_rows=[]
for key,a in TEST.items():
    for variant,T in [("raw",1.),("scaled",DEC[key]["temperature"])]:
        policy = DEC[key]['policies'][variant]
        agg=camera_statistics(a,T,policy);AGG[(*key,variant)]=agg
        local=calculate(agg)
        for i,cam in enumerate(CAMS):
            mask=CAM_INDEX==i
            per_camera_rows.append({"model":key[0],"seed":key[1],"variant":variant,"location":cam,
                "events":int(agg[i,0]),"true_categories_present":int(test_frame.loc[mask,"class_name"].nunique()),
                "small_camera":bool(agg[i,0]<CFG["small_camera_flag_below"]),
                "policy_status":policy['status'],"policy_reason":policy.get('reason',''),
                "has_acceptance":bool(agg[i,4]>0),
                "accepted":int(agg[i,4]),"accepted_errors":int(agg[i,5]),
                **{m:float(local[m][i]) for m in METRICS if m!="macro_f1"}})
        finite=local["accepted_error"][np.isfinite(local["accepted_error"])]
        diagnostic_rows.append({"model":key[0],"seed":key[1],"variant":variant,
            "mean_within_camera_ece":float(local["ece"].mean()),
            "mean_error_among_accepting_cameras":float(finite.mean()) if len(finite) else np.nan,
            "cameras_with_acceptance":len(finite),"total_cameras":C})
        for weighting,stats in [("image",agg),("equal_camera",agg/N_CAM[:,None])]:
            score={m:float(v[0]) for m,v in calculate(stats.sum(0)).items()}
            summary_rows.append({"model":key[0],"seed":key[1],"variant":variant,"weighting":weighting,
                "policy_status":policy['status'],"policy_reason":policy.get('reason',''),
                "accepted":int(agg[:,4].sum()),"accepted_errors":int(agg[:,5].sum()),**score})
            if weighting=="image":
                old=old_results.loc[old_results.model.eq(key[0])&old_results.seed.eq(key[1])&old_results.variant.eq(variant)]
                require(len(old)==1, "Missing/duplicate parent test-result row")
                for m in METRICS:
                    require(np.isclose(score[m],float(old.iloc[0][m]),rtol=1e-9,atol=1e-10,equal_nan=True),
                            f'Parent test metric differs: {key} {variant} {m}')
                require(int(old.iloc[0].accepted) == int(agg[:,4].sum())
                        and int(old.iloc[0].accepted_errors) == int(agg[:,5].sum())
                        and old.iloc[0].policy_status == policy['status'], 'Parent acceptance count/status differs')
per_camera=table("test_per_camera.csv",per_camera_rows)
weighted=table("weighting_comparison.csv",summary_rows)
table("within_camera_diagnostics.csv",diagnostic_rows)
means = summarize_audit_seeds(weighted)
table('weighting_seed_means.csv', means)
seed_abstention, camera_abstention = audit_abstention_tables(per_camera)
table('seed_acceptance_summary.csv', seed_abstention)
table('camera_abstention_summary.csv', camera_abstention)
# Reconcile the conditional seed summary with Notebook 04, retaining strict all-seed errors above.
for row in means.loc[means.weighting.eq('image')].itertuples(index=False):
    parent = parent_seed_summary.loc[parent_seed_summary.model.eq(row.model)&parent_seed_summary.variant.eq(row.variant)]
    require(len(parent) == 1, 'Missing/duplicate parent seed summary')
    for field in ['seed_count','accepted_error_contributing_seeds','abstaining_seeds','seeds_with_selected_policy']:
        require(int(parent.iloc[0][field]) == int(getattr(row,field)), f'Parent seed count differs: {field}')
    require(np.isclose(parent.iloc[0]['accepted_error_mean_among_accepting_seeds'],
                       row.accepted_error_mean_among_accepting_seeds, rtol=1e-9,atol=1e-10,equal_nan=True),
            'Parent conditional accepted-error seed mean differs')

display(means.round(4))
print("All parent Notebook 04 v3 event-weighted metrics and abstention counts reproduced.")


## 5. Is the calibration benefit spread across cameras?
For each model and camera, average the raw-minus-scaled change over all three seeds. Positive NLL/Brier/ECE gain means improvement; negative means deterioration. Count improved, worsened and approximately unchanged cameras, and plot the individual NLL changes. A camera with two events receives the same weight in an equal-camera mean as one with hundreds, but its observed rate is much less stable. The <30-event flag is descriptive, not an exclusion rule or significance threshold.

We also calculate which cameras contribute most to accepted mistakes. High event share and high error rate are different properties: counts and rates must be inspected together. We do not choose a favourable seed or remove difficult cameras.


In [ ]:
gain_rows=[];benefit_counts=[]
for model in MODELS:
    part=per_camera.loc[per_camera.model.eq(model)]
    for metric in ["nll","brier","ece"]:
        pairs=part.pivot(index=["seed","location"],columns="variant",values=metric)
        gains=(pairs.raw-pairs.scaled).groupby("location").mean().reindex(CAMS)
        for cam,gain in gains.items():
            n=int(N_CAM[np.where(CAMS==cam)[0][0]])
            gain_rows.append({"model":model,"location":cam,"metric":metric,"mean_seed_gain":float(gain),
                              "events":n,"small_camera":n<CFG["small_camera_flag_below"]})
        benefit_counts.append({"model":model,"metric":metric,"cameras":C,
            "improved":int((gains>1e-10).sum()),"worsened":int((gains< -1e-10).sum()),
            "approximately_unchanged":int((gains.abs()<=1e-10).sum()),
            "equal_camera_mean_gain":float(gains.mean())})
gains_by_camera=table("camera_calibration_gains.csv",gain_rows)
benefit_counts=table("camera_benefit_counts.csv",benefit_counts)
error_counts=per_camera[["model","seed","variant","location","events","accepted","accepted_errors","policy_status","policy_reason"]].copy()
error_counts["share_of_all_accepted_errors"]=error_counts.accepted_errors/error_counts.groupby(["model","seed","variant"]).accepted_errors.transform("sum").replace(0,np.nan)
table("accepted_error_concentration.csv",error_counts)
display(benefit_counts.round(4))
fig,axes=plt.subplots(2,1,figsize=(13,7),sharex=True)
for ax,model in zip(axes,["mobilenet_v3_large","efficientnet_b0"]):
    g=gains_by_camera.loc[gains_by_camera.model.eq(model)&gains_by_camera.metric.eq("nll")].set_index("location").reindex(CAMS)
    colors=["tab:gray" if small else "tab:blue" for small in g.small_camera]
    ax.bar(range(C),g.mean_seed_gain,color=colors)
    ax.axhline(0,color="black",lw=1);ax.set(title=model,ylabel="Raw − scaled NLL")
axes[-1].set(xticks=range(C),xticklabels=CAMS,xlabel="Camera ID (gray: fewer than 30 events)")
fig.tight_layout();fig.savefig(REPORT/"nll_gains_by_camera.png",dpi=160);plt.show();plt.close(fig)


## 6. Paired camera bootstrap: weightings and direct difference in gains
All comparisons use the same resampled camera multiplicities for both architectures, all seeds and both variants. For image weighting, sum the sampled cameras' event statistics. For equal-camera weighting, first divide each camera's statistics by its event count; sum those normalized statistics with the same multiplicities. Thus an included camera has equal total weight per occurrence.

Within each bootstrap draw, compute the metric for each trained run and average over the three fixed seeds. For loss metric $L$:

$$G_m=L_{m,raw}-L_{m,scaled},\qquad D=G_{EfficientNet}-G_{MobileNet}.$$

$G_m>0$ indicates calibration improvement. $D>0$ means EfficientNet improved more; $D<0$ means MobileNet improved more. The paired percentile interval for $D$ directly addresses the proposal's architectural difference question. Separate intervals showing improvement for both models do not establish a difference between their improvements. A model with lower final NLL need not have the larger gain.

We report intervals for seed-averaged metrics and gains under both weighting schemes. These condition on trained models, calibration/policy fitting and one test-camera split. They do not estimate refitting uncertainty or correct detector overlap, small calibration-camera diversity, or selection optimism. The actual number and event sizes of test cameras are reported; they limit generalization. Intervals for ECE remain bin-dependent. No p-values or multiple-testing-adjusted claims are produced.

The parent event-weighted primary NLL comparison is reproduced and checked first. A mismatch
stops the audit. Equal-camera weighting then extends that analysis without changing any fitted
model, temperature, policy or data partition. This does not provide a second independent test.

These are the **2,000 test-camera** resamples inherited from Notebook 04, not its 10,000
policy-camera diagnostic resamples. The 2.5th and 97.5th percentiles of finite test replicates
form the descriptive intervals; valid/total counts show how many draws contributed. An
all-seed accepted-error mean remains undefined in a draw if any seed has no acceptance;
the interval does not silently switch to a changing subset of seeds. If many draws are
undefined, the remaining conditional interval needs particular caution. If none is valid,
there is no interval to interpret.

An accepted-error interval of $[0,0]$ can occur when resampling only observed correct accepted
predictions. It does not establish zero population error or remove uncertainty about unseen
cameras. This audit supplies no replacement confidence bound for policy selection. Finite
camera-bootstrap intervals remain approximate with few cameras. More resamples do not create
additional independent sites, and reusing cameras across three seeds does not create new sites.


In [ ]:
rng=np.random.default_rng(CFG["bootstrap_seed"])
W=rng.multinomial(C,np.full(C,1/C),size=CFG["bootstrap_repeats"]).astype(float)
BOOT={};interval_rows=[];contrast_rows=[]
def interval(v):
    finite=np.asarray(v)[np.isfinite(v)]
    if not len(finite):return np.nan,np.nan,0
    return float(np.quantile(finite,.025)),float(np.quantile(finite,.975)),len(finite)
def point(model,variant,weighting,metric):
    x=weighted.loc[weighted.model.eq(model)&weighted.variant.eq(variant)&weighted.weighting.eq(weighting),metric]
    require(len(x)==len(SEEDS), "Missing seeds in audit point estimate")
    return float(x.mean()) if x.notna().all() else np.nan
def strict_mean(arrays):
    return np.mean(np.stack(arrays), axis=0)


def calibration_gain_comparisons(boot, result_table, seeds):
    if list(seeds) != [42, 123, 2026]:
        raise ValueError('The planned primary comparison requires all three paired seeds')
    gains = {}
    points = {}
    rows = []
    for model in ['mobilenet_v3_large', 'efficientnet_b0']:
        part = result_table.loc[result_table.model.eq(model)]
        pair = part.pivot(index='seed', columns='variant', values='nll').reindex(seeds)
        if pair[['raw', 'scaled']].isna().any().any():
            raise ValueError('Missing paired NLL scores in the primary calibration-gain comparison')
        points[model] = float((pair.raw-pair.scaled).mean())
        gains[model] = strict_mean([boot[(model, seed, 'raw')]['nll']-
                                  boot[(model, seed, 'scaled')]['nll'] for seed in seeds])
        lo, hi, valid = interval(gains[model])
        rows.append({'comparison':model+' NLL gain (raw minus scaled; mean seeds)', 'metric':'nll',
                     'primary_metric':True, 'estimate':points[model], 'ci_low':lo, 'ci_high':hi,
                     'valid_replicates':valid, 'total_replicates':len(gains[model])})
    difference = gains['efficientnet_b0']-gains['mobilenet_v3_large']
    lo, hi, valid = interval(difference)
    rows.append({'comparison':'EfficientNet NLL gain minus MobileNet NLL gain (mean paired seeds)',
                 'metric':'nll', 'primary_metric':True,
                 'estimate':points['efficientnet_b0']-points['mobilenet_v3_large'],
                 'ci_low':lo, 'ci_high':hi, 'valid_replicates':valid, 'total_replicates':len(difference)})
    return pd.DataFrame(rows)


def verify_primary_gains(recomputed, parent):
    require(len(parent) == 3 and len(recomputed) == 3, 'Expected three primary NLL gain rows')
    require(set(parent.comparison) == set(recomputed.comparison)
            and not parent.comparison.duplicated().any(), 'Primary NLL comparison identities differ')
    rows = []
    for row in recomputed.itertuples(index=False):
        prior = parent.loc[parent.comparison.eq(row.comparison)].iloc[0]
        require(prior['metric'] == row.metric == 'nll'
                and isinstance(prior['primary_metric'], (bool,np.bool_)) and bool(prior['primary_metric']),
                'Wrong parent primary metric')
        for field in ['estimate','ci_low','ci_high']:
            require(np.isclose(float(prior[field]),getattr(row,field),rtol=1e-9,atol=1e-10,equal_nan=True),
                    f'Primary Notebook 04 NLL comparison differs: {row.comparison} {field}')
        for field in ['valid_replicates','total_replicates']:
            require(float(prior[field]) == float(getattr(row,field)), 'Primary bootstrap replicate counts differ')
        rows.append({'comparison':row.comparison,'metric':'nll','matched':True,
                     'parent_estimate':float(prior['estimate']),'audit_estimate':row.estimate,
                     'ci_low':row.ci_low,'ci_high':row.ci_high,
                     'valid_replicates':row.valid_replicates,'total_replicates':row.total_replicates})
    return pd.DataFrame(rows)


# First reproduce the parent event-weighted primary comparison, with identical sorted
# camera IDs, shared draws, all seeds and fixed predictions. This is not a new test.
for key,agg in AGG.items():
    BOOT[(*key,'image')] = calculate(W@agg)
image_boot = {key[:3]:value for key,value in BOOT.items()}
primary_recomputed = calibration_gain_comparisons(image_boot,
    weighted.loc[weighted.weighting.eq('image')], SEEDS)
primary_consistency = table('notebook04_gain_consistency.csv', verify_primary_gains(primary_recomputed,parent_gains))
# Only after the primary comparison agrees, add the equal-camera sensitivity.
for key,agg in AGG.items():
    BOOT[(*key,'equal_camera')] = calculate(W@(agg/N_CAM[:,None]))
# Average across all three seeds; undefined risks remain undefined (no silent seed dropping).
SEED_BOOT={}
for model in MODELS:
    for variant in ["raw","scaled"]:
        for weighting in ["image","equal_camera"]:
            for metric in METRICS:
                v=np.mean(np.stack([BOOT[(model,s,variant,weighting)][metric] for s in SEEDS]),axis=0)
                SEED_BOOT[(model,variant,weighting,metric)]=v
                lo,hi,valid=interval(v)
                interval_rows.append({"model":model,"variant":variant,"weighting":weighting,"metric":metric,
                    "estimate":point(model,variant,weighting,metric),"ci_low":lo,"ci_high":hi,
                    "valid_replicates":valid,"total_replicates":len(W)})
for weighting in ["image","equal_camera"]:
    for metric in ["nll","brier","ece"]:
        gains={};gain_points={}
        for model in MODELS:
            gains[model]=SEED_BOOT[(model,"raw",weighting,metric)]-SEED_BOOT[(model,"scaled",weighting,metric)]
            gain_points[model]=point(model,"raw",weighting,metric)-point(model,"scaled",weighting,metric)
            lo,hi,valid=interval(gains[model])
            contrast_rows.append({"comparison":model+" calibration gain (raw minus scaled)","weighting":weighting,
                "metric":metric,"estimate":gain_points[model],"ci_low":lo,"ci_high":hi,
                "valid_replicates":valid,"total_replicates":len(W)})
        difference=gains["efficientnet_b0"]-gains["mobilenet_v3_large"]
        lo,hi,valid=interval(difference)
        contrast_rows.append({"comparison":"EfficientNet gain minus MobileNet gain","weighting":weighting,
            "metric":metric,"estimate":gain_points["efficientnet_b0"]-gain_points["mobilenet_v3_large"],
            "ci_low":lo,"ci_high":hi,"valid_replicates":valid,"total_replicates":len(W)})
intervals=table("weighted_camera_bootstrap_intervals.csv",interval_rows)
contrasts=table("calibration_gain_contrasts.csv",contrast_rows)
display(contrasts.round(5))
# Algebra checks: equal-camera NLL gain is the mean of per-camera NLL gains.
for model in MODELS:
    direct=gains_by_camera.loc[gains_by_camera.model.eq(model)&gains_by_camera.metric.eq("nll"),"mean_seed_gain"].mean()
    require(np.isclose(direct,point(model,"raw","equal_camera","nll")-point(model,"scaled","equal_camera","nll"),rtol=1e-9,atol=1e-10),
            'Equal-camera NLL gain does not equal the mean per-camera gain')


## 7. Influence of individual test cameras
Leave-one-camera-out analysis recalculates the mean-seed NLL gain with each camera omitted in turn, using the original fixed predictions and decisions. This is an influence diagnostic, not retraining, refitting, a new split search, or justification to discard locations. Compare the range of these gains and whether their direction changes. A conclusion that changes on omitting one camera is sensitive to that observed location.

This cannot tell us whether excluding a calibration camera would change the temperature. That would be a distinct refitting experiment. We deliberately keep the original fitting procedure fixed here.

For each omitted location, also compute EfficientNet gain minus MobileNet gain using that same
omission across all three seeds. Report the contrast range, sign flips, direction changes and
camera IDs. Zero is treated as approximately unchanged at the original 1e-10 descriptive tolerance.

The minimum-to-maximum leave-one-camera-out contrast range is an influence summary, not a
confidence interval. A reversal identifies sensitivity to an observed camera; it does not
establish statistical significance, identify its cause or authorize removing that camera.
Both weighting schemes retain the full original camera set for their main estimates.


In [ ]:
loo_rows=[]
for weighting in ["image","equal_camera"]:
    for model in MODELS:
        for i,cam in enumerate(CAMS):
            local={}
            for variant in ["raw","scaled"]:
                values=[]
                for seed in SEEDS:
                    a=AGG[(model,seed,variant)]
                    stats=a if weighting=="image" else a/N_CAM[:,None]
                    values.append(float(calculate(stats.sum(0)-stats[i])["nll"][0]))
                local[variant]=float(np.mean(values))
            loo_rows.append({"model":model,"weighting":weighting,"omitted_camera":cam,
                "omitted_events":int(N_CAM[i]),"nll_gain":local["raw"]-local["scaled"],
                "full_test_nll_gain":point(model,"raw",weighting,"nll")-point(model,"scaled",weighting,"nll")})
loo=table("leave_one_camera_out_nll_gains.csv",loo_rows)
loo_summary=loo.groupby(["model","weighting"]).agg(minimum_gain=("nll_gain","min"),
    maximum_gain=("nll_gain","max"),full_test_gain=("full_test_nll_gain","first")).reset_index()
table("leave_one_camera_out_summary.csv",loo_summary)
display(loo_summary.round(5))
def architecture_camera_influence(leave_one_out):
    rows = []
    def direction(value):
        return 1 if value > 1e-10 else -1 if value < -1e-10 else 0
    for (weighting,camera), group in leave_one_out.groupby(['weighting','omitted_camera']):
        require(len(group) == 2 and set(group.model) == set(MODELS) and not group.model.duplicated().any(),
                'Camera influence requires the same omitted camera in both architectures')
        pair = group.set_index('model')
        require(pair.omitted_events.nunique() == 1, 'Architecture influence used different omitted events')
        mobile = float(pair.loc['mobilenet_v3_large','nll_gain'])
        efficient = float(pair.loc['efficientnet_b0','nll_gain'])
        full = float(pair.loc['efficientnet_b0','full_test_nll_gain']-
                     pair.loc['mobilenet_v3_large','full_test_nll_gain'])
        difference = efficient-mobile
        require(np.isfinite([mobile,efficient,full,difference]).all(), 'Nonfinite camera influence NLL')
        rows.append({'weighting':weighting,'omitted_camera':str(camera),
            'omitted_events':int(pair.omitted_events.iloc[0]),
            'mobilenet_nll_gain':mobile,'efficientnet_nll_gain':efficient,
            'nll_gain_difference':difference,'full_test_nll_gain_difference':full,
            'change_from_full':difference-full,'full_direction':direction(full),
            'omitted_direction':direction(difference),
            'sign_flip':direction(full)*direction(difference) < 0,
            'direction_changed':direction(full) != direction(difference)})
    detail = pd.DataFrame(rows)
    summary = []
    for weighting, group in detail.groupby('weighting'):
        require(group.full_test_nll_gain_difference.nunique() == 1, 'Inconsistent full-test architecture contrast')
        summary.append({'weighting':weighting,'cameras':len(group),
            'full_test_difference':float(group.full_test_nll_gain_difference.iloc[0]),
            'minimum_omitted_camera_difference':float(group.nll_gain_difference.min()),
            'maximum_omitted_camera_difference':float(group.nll_gain_difference.max()),
            'sign_flips':int(group.sign_flip.sum()),'direction_changes':int(group.direction_changed.sum()),
            'sign_flip_camera_ids':json.dumps(sorted(group.loc[group.sign_flip,'omitted_camera'].tolist()))})
    return detail, pd.DataFrame(summary)


architecture_influence, architecture_influence_summary = architecture_camera_influence(loo)
table('architecture_camera_influence.csv', architecture_influence)
table('architecture_camera_influence_summary.csv', architecture_influence_summary)
display(architecture_influence_summary.round(5))

fig,axes=plt.subplots(1,2,figsize=(12,4))
for ax,weighting in zip(axes,["image","equal_camera"]):
    for i,model in enumerate(["mobilenet_v3_large","efficientnet_b0"]):
        r=contrasts.loc[contrasts.weighting.eq(weighting)&contrasts.metric.eq("nll")&contrasts.comparison.eq(model+" calibration gain (raw minus scaled)")].iloc[0]
        ax.plot([r.ci_low,r.ci_high],[i,i],lw=2)
        ax.scatter([r.estimate],[i],s=35)
    ax.axvline(0,color="gray",ls="--")
    ax.set(title=weighting,yticks=[0,1],yticklabels=["MobileNet","EfficientNet"],xlabel="NLL gain (raw − scaled), camera-bootstrap interval")
fig.tight_layout();fig.savefig(REPORT/"nll_gains_by_weighting.png",dpi=160);plt.show();plt.close(fig)


## 8. Generate the v3 report from actual audited outputs
Descriptions use the actual v3 camera counts, selected checkpoint stages, policy statuses,
contributing/abstaining seeds, consistency results and influence diagnostics. No v1 target outcome
or fixed camera count is assumed. Models and policy decisions remain fixed, and no crop review is
claimed. Recheck the parent output manifest, completion record and prediction hashes before
writing the audit completion record. NLL is primary; supporting outcomes remain sensitivity analyses.


In [ ]:
def plain(frame):
    return '```text\n'+frame.to_string(index=False,float_format=lambda value:f'{value:.5f}')+'\n```'

policy_outcomes = table('frozen_policy_outcomes.csv', [
    {'model':model,'seed':seed,'variant':variant,'status':decision['status'],
     'reason':decision.get('reason',''),'threshold':decision['threshold']}
    for (model,seed), run in DEC.items() for variant,decision in run['policies'].items()])
policy_status_counts = policy_outcomes.groupby(['model','variant','status']).size().rename('seeds').reset_index()
notes = f"""# Camera robustness audit — v3 actual results
Generated: {now()}. Experiment: {EXPERIMENT_ID}; training: {TRAIN_ID}.
Parent evaluation hash: {completion['evaluation_sha256']}; frozen decisions: {completion['decisions_sha256']}.

## Scope and provenance
This sensitivity analysis uses the existing v3 predictions, frozen temperatures and acceptance policies.
It does not retrain models, refit temperatures/thresholds, resplit data, download or inspect images,
or run new model inference. Corrected selected checkpoints may be head-only or fine-tuned; stage/epoch
and checkpoint digests are bound through the Notebook 03 index and Notebook 04 evaluation records.
There are {len(TEST)} trained runs, {len(test_frame)} test events, {C} represented test cameras,
and {len(SEEDS)} paired training seeds. Manual crop verification was not performed; possible crop-label
mismatches remain a limitation. This is a v3 sensitivity analysis informed by the earlier study,
not a fresh independent test or proof of preregistration. Methods must not be revised using v3 outcomes.

## Camera and category representation
{plain(concentration)}
{plain(class_shares.reset_index())}
Concentration-equivalent cameras is 1/sum(event_share squared), a count-concentration descriptor,
not a statistical effective sample size. Per-category concentration is recorded separately.
The audit describes the already-selected sample; it does not repair camera concentration or establish
that class/background changes caused generalization failures.

## Frozen policy outcomes and abstention
{plain(policy_status_counts)}
{plain(seed_abstention)}
Each seed and camera with no accepted predictions has undefined accepted error, never zero error.
The camera_abstention_summary.csv and test_per_camera.csv files identify the affected cameras and seeds.
Thresholds follow Notebook 04's frozen binomial-bound selection and support requirements.
Camera uncertainty is diagnostic only: zero observed policy errors or an unavailable camera estimate
can coexist with a selected threshold. No-threshold outcomes are not treated as classification failure.
Classification/calibration metrics remain meaningful when no policy accepts.
No camera-weighted result certifies a new threshold or provides guaranteed camera-aware error control.

## Event and equal-camera weighting
{plain(means)}
Strict all-seed accepted-error means remain undefined if any seed has no acceptance. The separate
conditional mean among accepting seeds reports contributor counts and seed IDs explicitly, matching
Notebook 04's convention. All seeds contribute to coverage and classification/calibration summaries.
Event weighting describes this observed event mixture. Equal-camera weighting draws an observed
camera uniformly and an event uniformly within it; it does not imply representativeness of future sites.
Equal-camera pooled ECE/F1 are computed from camera-normalized additive statistics, not averages of
camera ECE/F1. Weighted accepted error is weighted mistakes divided by weighted accepted observations,
not the mean conditional error among accepting cameras. Those local diagnostics are labelled separately.

## Primary Notebook 04 NLL comparison verified
{plain(primary_consistency)}
Event-weighted NLL gains and the architecture difference, including paired bootstrap endpoints and
replicate counts, agree with Notebook 04. This reproduction is a consistency check, not independent
confirmation. NLL gain is raw minus scaled; the architecture contrast is EfficientNet gain minus
MobileNet gain. Positive means EfficientNet improved more; negative means MobileNet improved more.
The equal-camera contrast below is a sensitivity analysis. Brier/ECE are supporting outcomes.

## Camera-specific benefit and weighting sensitivity
{plain(benefit_counts)}
{plain(contrasts)}
Improved/worsened camera counts are descriptive; small-camera estimates are unstable. The
<{CFG['small_camera_flag_below']}-event flag is not an exclusion rule or significance threshold.
Camera IDs represent dataset locations, not verified independent habitats or spatial distances.

## Influence of individual cameras
{plain(loo_summary)}
{plain(architecture_influence_summary)}
Leave-one-camera-out calculations keep predictions and policies fixed. They show whether omitting
a location changes each model's NLL gain or reverses the comparison between architectures.
A sign flip is descriptive influence, not evidence of significance or a reason to exclude the camera.
The full original test-camera set remains the reported evaluation population.

## Uncertainty and limitations
All analyses inherit {CFG['bootstrap_repeats']} shared whole-camera bootstrap draws with seed
{CFG['bootstrap_seed']} and {CFG['ece_bins']} confidence bins from Notebook 04. They average the same
{len(SEEDS)} seed-specific scores inside each draw. Percentile intervals condition on fitted models,
temperatures, policies and this camera split. They exclude refitting/split uncertainty, are exploratory
across supporting outcomes, and do not establish equivalence when they cross zero. Undefined
bootstrap risks remain missing and valid/total replicate counts are reported. Equal-camera weighting
increases the influence of sparsely observed sites; ECE is bin-dependent. Detector pretraining overlap
with Caltech and different classifier pretraining recipes remain limitations. The actual Notebook 04
policy outcomes are retained above; no v1 target-failure conclusion is copied into this report.

## Output files
Representation: camera_concentration_by_split.csv, camera_event_counts.csv, class_camera_counts.csv,
category_camera_concentration.csv, class_distribution_by_split.csv. Performance: test_per_camera.csv,
weighting_comparison.csv, weighting_seed_means.csv, within_camera_diagnostics.csv. Abstention:
seed_acceptance_summary.csv, camera_abstention_summary.csv, frozen_policy_outcomes.csv. Calibration:
notebook04_gain_consistency.csv, camera_calibration_gains.csv, camera_benefit_counts.csv,
weighted_camera_bootstrap_intervals.csv, calibration_gain_contrasts.csv. Influence:
leave_one_camera_out_nll_gains.csv, leave_one_camera_out_summary.csv,
architecture_camera_influence.csv, architecture_camera_influence_summary.csv.
The parent evaluation and its cached predictions remain unchanged.
"""
write(REPORT/'CAMERA_AUDIT_METHODS_AND_RESULTS.md', notes)
verify_parent_unchanged()
for entry in input_refs:
    verified_bytes(safe(entry['file']), entry['sha256'])
write(REPORT/'completion.json', jtext({'time_utc':now(),'experiment_id':EXPERIMENT_ID,'training_id':TRAIN_ID,
    'audit_protocol_sha256':sha(REPORT/'audit_protocol.json'),
    'parent_completion_sha256':PARENT_COMPLETION_SHA,'parent_evaluation_sha256':completion['evaluation_sha256'],
    'decisions_sha256':completion['decisions_sha256'],'test_cameras':C,'test_events':len(test_frame),
    'runs':len(TEST),'seeds':SEEDS,'primary_notebook04_results_verified':True,
    'original_evaluation_unchanged':True,'refitting_performed':False,'new_model_inference_performed':False,
    'outputs':{path.name:sha(path) for path in sorted(REPORT.iterdir())
               if path.is_file() and path.name != 'completion.json'}}))
print('V3 audit complete:', REPORT)
print('Parent evaluation and original prediction hashes verified unchanged.')
